# 09 Full Experiment Suite Runner

Run the complete experiment suite (all core methods, sweeps, and report generation) from inside a notebook.

In [1]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


Repo root: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv
Using src dir: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/src
Using suite output dir: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/experiment_suite


In [2]:
from pathlib import Path
import os
import time

from cytof_archetypes.experiments import load_suite_config, run_experiment_suite

CONFIG_PATH = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
assert CONFIG_PATH.exists(), f'Missing config: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH)

# Optional: override output dir without editing YAML.
# os.environ['CYTOF_SUITE_OUTPUT_DIR'] = str(REPO_ROOT / 'outputs' / 'experiment_suite')

Using config: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/configs/experiment_suite.yaml


In [3]:
cfg = load_suite_config(CONFIG_PATH)
env_out = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
if env_out:
    cfg['output_dir'] = env_out

# Speed controls for quick testing:
DOWNSAMPLE_FACTOR = 1  # e.g. 5 keeps ~20% of cells, 10 keeps ~10%
if DOWNSAMPLE_FACTOR and DOWNSAMPLE_FACTOR > 1:
    cfg.setdefault('dataset', {})['downsample_factor'] = int(DOWNSAMPLE_FACTOR)
else:
    cfg.setdefault('dataset', {}).pop('downsample_factor', None)
    cfg.setdefault('dataset', {}).pop('downsample_fraction', None)

# Verbose execution prints and progress bars:
cfg['show_progress'] = True
cfg['show_run_logs'] = True

print('Resolved output_dir:', cfg['output_dir'])
print('Resolved default device:', cfg.get('resolved_device', cfg.get('device')))
for _m in ['deterministic_archetypal_ae', 'probabilistic_archetypal_ae', 'ae', 'vae']:
    print(f"  {_m}:", cfg.get('methods', {}).get(_m, {}).get('device'))
print('Methods:', list(cfg.get('methods', {}).keys()))
print('Seeds:', cfg.get('seeds', []))
print('K sweep:', cfg.get('sweeps', {}).get('k_values', []))
print('Latent sweep:', cfg.get('sweeps', {}).get('latent_dims', []))
print('Downsample factor:', cfg.get('dataset', {}).get('downsample_factor', 1))
print('show_progress:', cfg.get('show_progress'), 'show_run_logs:', cfg.get('show_run_logs'))

Resolved output_dir: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/experiment_suite
Resolved default device: mps
  deterministic_archetypal_ae: mps
  probabilistic_archetypal_ae: mps
  ae: mps
  vae: mps
Methods: ['nmf', 'classical_archetypes', 'deterministic_archetypal_ae', 'probabilistic_archetypal_ae', 'ae', 'vae']
Seeds: [13, 17, 23]
K sweep: [8, 10, 12, 14, 16]
Latent sweep: [8, 10, 12, 14, 16]
Downsample factor: 1
show_progress: True show_run_logs: True


In [4]:
RUN_FULL = True  # set to False to skip execution
if RUN_FULL:
    t0 = time.time()
    out_dir = run_experiment_suite(cfg)
    dt = time.time() - t0
    print(f'Completed full suite in {dt/60:.2f} minutes')
    print('Output directory:', out_dir)
else:
    out_dir = Path(cfg['output_dir'])
    print('Execution skipped. Expected output directory:', out_dir)

Suite phases:   0%|          | 0/12 [00:00<?, ?phase/s]

[suite-phase] 1/12: prepare_data
[suite-phase] 2/12: run_core_benchmark (all methods × dims × seeds)


Core benchmark runs:   0%|          | 0/90 [00:00<?, ?run/s]

[core-run 1/90] START method=nmf dim=8 seed=13 device=cpu(sklearn)
[core-run 2/90] START method=nmf dim=8 seed=17 device=cpu(sklearn)
[core-run 3/90] START method=nmf dim=8 seed=23 device=cpu(sklearn)
[core-run 4/90] START method=nmf dim=10 seed=13 device=cpu(sklearn)
[core-run 5/90] START method=nmf dim=10 seed=17 device=cpu(sklearn)
[core-run 6/90] START method=nmf dim=10 seed=23 device=cpu(sklearn)
[core-run 7/90] START method=nmf dim=12 seed=13 device=cpu(sklearn)
[core-run 8/90] START method=nmf dim=12 seed=17 device=cpu(sklearn)
[core-run 9/90] START method=nmf dim=12 seed=23 device=cpu(sklearn)
[core-run 10/90] START method=nmf dim=14 seed=13 device=cpu(sklearn)
[core-run 11/90] START method=nmf dim=14 seed=17 device=cpu(sklearn)
[core-run 12/90] START method=nmf dim=14 seed=23 device=cpu(sklearn)
[core-run 13/90] START method=nmf dim=16 seed=13 device=cpu(sklearn)
[core-run 14/90] START method=nmf dim=16 seed=17 device=cpu(sklearn)
[core-run 15/90] START method=nmf dim=16 seed=

deterministic_archetypal_ae dim=8 seed=13 [epoch]:   0%|          | 0/80 [00:00<?, ?epoch/s]

...

[core-run 33/90] DONE  method=deterministic_archetypal_ae dim=8 seed=23 val_mse=0.427255 test_mse=0.427391 elapsed=249.7s
[core-run 34/90] START method=deterministic_archetypal_ae dim=10 seed=13 device=mps


deterministic_archetypal_ae dim=10 seed=13 [epoch]:   0%|          | 0/80 [00:00<?, ?epoch/s]

[core-run 34/90] DONE  method=deterministic_archetypal_ae dim=10 seed=13 val_mse=0.359148 test_mse=0.359362 elapsed=224.5s
[core-run 35/90] START method=deterministic_archetypal_ae dim=10 seed=17 device=mps


deterministic_archetypal_ae dim=10 seed=17 [epoch]:   0%|          | 0/80 [00:00<?, ?epoch/s]

KeyboardInterrupt: 

In [8]:
import pandas as pd

out_dir = Path(cfg['output_dir'])
tables_dir = out_dir / 'tables'
reports_dir = out_dir / 'reports'
plots_dir = out_dir / 'plots'

required_tables = [
    'fit_vs_complexity_summary.csv',
    'deconvolution_quality_summary.csv',
    'component_marker_profiles.csv',
    'component_marker_enrichment.csv',
    'deterministic_vs_probabilistic_summary.csv',
    'per_class_method_metrics.csv',
    'fit_vs_interpretability.csv',
    'k_selection_summary.csv',
    'class_component_means.csv',
    'per_cell_weight_entropy.csv',
]

status = []
for name in required_tables:
    path = tables_dir / name
    status.append({'table': name, 'exists': path.exists(), 'path': str(path)})
status_df = pd.DataFrame(status)
display(status_df)
print('All required tables present:', bool(status_df['exists'].all()))

,table,exists,path
0,fit_vs_complexity_summary.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
1,deconvolution_quality_summary.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
2,component_marker_profiles.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
3,component_marker_enrichment.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
4,deterministic_vs_probabilistic_summary.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
5,per_class_method_metrics.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
6,fit_vs_interpretability.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
7,k_selection_summary.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
8,class_component_means.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...
9,per_cell_weight_entropy.csv,True,/Users/ronguy/Dropbox/Work/CyTOF/Experiments/P...


All required tables present: True


In [9]:
manifest_path = reports_dir / 'artifact_manifest.json'
if manifest_path.exists():
    import json
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('Manifest summary keys:', sorted(manifest.keys()))
    print('Number of tables:', len(manifest.get('tables', [])))
    print('Number of plots:', len(manifest.get('plots', [])))
else:
    print('Missing artifact manifest:', manifest_path)

Manifest summary keys: ['has_auxiliary_results', 'has_secondary_dataset_results', 'n_component_profile_rows', 'n_deconvolution_rows', 'n_enrichment_rows', 'n_k_selection_rows', 'n_rare_rows', 'notebook_output_dir', 'notebooks', 'plots', 'tables']
Number of tables: 13
Number of plots: 18


In [10]:
k_path = tables_dir / 'k_selection_summary.csv'
if k_path.exists():
    k_df = pd.read_csv(k_path)
    display(k_df.sort_values(['method', 'k']).head(30))
else:
    print('Missing:', k_path)

,method,k,seed,val_mse,test_mse,rare_class_error,interpretability_score,component_redundancy,fit_score,rare_score,interp_score,redundancy_score,k_selection_score,recommended_k
0,classical_archetypes,8,17.666667,0.578091,0.562664,0.954840,0.844545,0.243274,0.000000,0.000000,0.000000,0.790220,0.197555,16
1,classical_archetypes,10,17.666667,0.544397,0.528966,0.885455,0.856786,0.240747,0.417990,0.437897,1.000000,1.000000,0.713972,16
2,classical_archetypes,12,17.666667,0.525319,0.509624,0.827529,0.854437,0.250995,0.654659,0.803482,0.808141,0.149340,0.603905,16
3,classical_archetypes,14,17.666667,0.507946,0.493661,0.803176,0.848613,0.252795,0.870167,0.957178,0.332355,0.000000,0.539925,16
4,classical_archetypes,16,17.666667,0.497481,0.484542,0.796391,0.848390,0.242786,1.000000,1.000000,0.314117,0.830734,0.786213,16
5,deterministic_archetypal_ae,8,17.666667,0.683666,0.664971,1.036830,0.844095,0.159618,0.852447,1.000000,0.000000,0.000000,0.463112,12
6,deterministic_archetypal_ae,10,17.666667,0.699595,0.682859,1.057278,0.887489,0.102259,0.514320,0.829025,1.000000,0.793157,0.784125,12
7,deterministic_archetypal_ae,12,17.666667,0.676716,0.658433,1.057456,0.878084,0.110480,1.000000,0.827539,0.783269,0.679469,0.822569,12
8,deterministic_archetypal_ae,14,17.666667,0.723823,0.703777,1.156431,0.862147,0.094835,0.000000,0.000000,0.416007,0.895813,0.327955,12
9,deterministic_archetypal_ae,16,17.666667,0.697131,0.678252,1.139593,0.884215,0.087300,0.566615,0.140786,0.924556,1.000000,0.657989,12
